# 01 — Attention Mechanism

Step-by-step implementation of self-attention in NumPy, multi-head attention, and positional encoding visualization.

## 1. Self-Attention from Scratch

Self-attention allows each token in a sequence to attend to every other token, computing a weighted combination of values based on relevance.

The mechanism uses three projections:
- **Query (Q):** What am I looking for?
- **Key (K):** What do I contain?
- **Value (V):** What information do I provide?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def self_attention(Q, K, V):
    """
    Compute self-attention.
    
    Args:
        Q: Query matrix (seq_len, d_k)
        K: Key matrix (seq_len, d_k)
        V: Value matrix (seq_len, d_v)
    
    Returns:
        output: Attention output (seq_len, d_v)
        weights: Attention weights (seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    
    # Step 1: Compute attention scores
    scores = np.dot(Q, K.T) / np.sqrt(d_k)
    
    # Step 2: Apply softmax to get attention weights
    weights = softmax(scores)
    
    # Step 3: Weighted sum of values
    output = np.dot(weights, V)
    
    return output, weights

### 1.1 Example: Simple 4-Token Sequence

In [ ]:
# Simulate embeddings for 4 tokens: "The", "cat", "sat", "down"
seq_len = 4
d_k = 8  # embedding dimension

# Random embeddings (normally these come from an embedding layer)
embeddings = np.random.randn(seq_len, d_k)

# Create Q, K, V projections (in real models, these are learned weights)
W_Q = np.random.randn(d_k, d_k) * 0.1
W_K = np.random.randn(d_k, d_k) * 0.1
W_V = np.random.randn(d_k, d_k) * 0.1

Q = embeddings @ W_Q
K = embeddings @ W_K
V = embeddings @ W_V

output, weights = self_attention(Q, K, V)

print(f"Input shape: {embeddings.shape}")
print(f"Q shape: {Q.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"Output shape: {output.shape}")

In [ ]:
# Visualize attention weights
tokens = ["The", "cat", "sat", "down"]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(weights, cmap="Blues")

ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens)
ax.set_yticklabels(tokens)
ax.set_xlabel("Key (attending to)")
ax.set_ylabel("Query (looking from)")
ax.set_title("Self-Attention Weights")

# Add text annotations
for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f"{weights[i, j]:.2f}", ha="center", va="center", fontsize=10)

plt.colorbar(im)
plt.tight_layout()
plt.show()

### 1.2 Verifying Row Sums

Each row of the attention weights should sum to 1.0 (it's a probability distribution).

In [ ]:
print("Row sums (should all be 1.0):")
print(np.round(weights.sum(axis=1), 6))

## 2. Why Scale by √d_k?

Without scaling, large dimensionality leads to large dot products, pushing softmax into regions with tiny gradients.

In [ ]:
# Compare scaled vs unscaled attention
large_dk = 64
Q_large = np.random.randn(4, large_dk)
K_large = np.random.randn(4, large_dk)

unscaled_scores = np.dot(Q_large, K_large.T)
scaled_scores = np.dot(Q_large, K_large.T) / np.sqrt(large_dk)

print(f"Unscaled scores range: [{unscaled_scores.min():.2f}, {unscaled_scores.max():.2f}]")
print(f"Scaled scores range: [{scaled_scores.min():.2f}, {scaled_scores.max():.2f}]")
print(f"\nUnscaled softmax (concentrated): {softmax(unscaled_scores)[0]}")
print(f"Scaled softmax (smoother): {softmax(scaled_scores)[0]}")

## 3. Multi-Head Attention

Multiple attention heads allow the model to attend to different aspects of the input simultaneously.

In [ ]:
def multi_head_attention(embeddings, n_heads):
    """
    Multi-head attention implementation.
    
    Args:
        embeddings: Input (seq_len, d_model)
        n_heads: Number of attention heads
    
    Returns:
        output: Concatenated head outputs (seq_len, d_model)
        all_weights: List of attention weight matrices
    """
    seq_len, d_model = embeddings.shape
    d_k = d_model // n_heads
    
    # Shared projection matrices (simplified)
    W_Q = np.random.randn(d_model, d_model) * 0.1
    W_K = np.random.randn(d_model, d_model) * 0.1
    W_V = np.random.randn(d_model, d_model) * 0.1
    W_O = np.random.randn(d_model, d_model) * 0.1
    
    Q = embeddings @ W_Q
    K = embeddings @ W_K
    V = embeddings @ W_V
    
    head_outputs = []
    all_weights = []
    
    for i in range(n_heads):
        # Each head gets a slice of Q, K, V
        start = i * d_k
        end = start + d_k
        
        Q_head = Q[:, start:end]
        K_head = K[:, start:end]
        V_head = V[:, start:end]
        
        head_out, head_weights = self_attention(Q_head, K_head, V_head)
        head_outputs.append(head_out)
        all_weights.append(head_weights)
    
    # Concatenate heads
    concatenated = np.concatenate(head_outputs, axis=-1)
    
    # Final linear projection
    output = concatenated @ W_O
    
    return output, all_weights

In [ ]:
# Test multi-head attention
d_model = 32
n_heads = 4

embeddings = np.random.randn(4, d_model)  # 4 tokens, 32-dim embeddings
output, all_weights = multi_head_attention(embeddings, n_heads)

print(f"Input: {embeddings.shape}")
print(f"Output: {output.shape}")
print(f"Number of heads: {len(all_weights)}")
print(f"Each head attention weights: {all_weights[0].shape}")

In [ ]:
# Visualize attention patterns across heads
tokens = ["The", "cat", "sat", "down"]
fig, axes = plt.subplots(1, n_heads, figsize=(15, 4))

for i, (ax, head_weights) in enumerate(zip(axes, all_weights)):
    im = ax.imshow(head_weights, cmap="Blues")
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens)
    ax.set_yticklabels(tokens)
    ax.set_title(f"Head {i + 1}")

plt.suptitle("Different Heads Learn Different Attention Patterns")
plt.tight_layout()
plt.show()

## 4. Positional Encoding

Self-attention is permutation-invariant — it doesn't know token order. Positional encoding fixes this.

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    """
    Generate sinusoidal positional encodings.
    
    Args:
        max_len: Maximum sequence length
        d_model: Embedding dimension
    
    Returns:
        Positional encodings (max_len, d_model)
    """
    PE = np.zeros((max_len, d_model))
    position = np.arange(0, max_len).reshape(-1, 1)
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    
    PE[:, 0::2] = np.sin(position * div_term)
    PE[:, 1::2] = np.cos(position * div_term)
    
    return PE

In [ ]:
# Visualize positional encodings
max_len = 50
d_model = 64

PE = sinusoidal_positional_encoding(max_len, d_model)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(PE, cmap="RdBu", aspect="auto")
ax.set_xlabel("Embedding Dimension")
ax.set_ylabel("Position")
ax.set_title("Sinusoidal Positional Encodings")
plt.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
# Show that positional encodings are unique per position
print("Cosine similarity between position 0 and others:")
for pos in [1, 5, 10, 25, 49]:
    sim = np.dot(PE[0], PE[pos]) / (np.linalg.norm(PE[0]) * np.linalg.norm(PE[pos]))
    print(f"  Position 0 vs {pos}: {sim:.4f}")

## 5. Building Intuition: What Attention Learns

Let's create a synthetic example to see how attention can capture relationships.

In [ ]:
# Simulate a simple syntactic relationship
# "The cat that chased the mouse sat down"
# We expect "cat" to attend to "sat" (subject-verb)

tokens = ["The", "cat", "that", "chased", "the", "mouse", "sat", "down"]
n_tokens = len(tokens)
d_k = 16

# Create embeddings where related words are closer
# (In real models, this emerges from training)
np.random.seed(123)
embeddings = np.random.randn(n_tokens, d_k)

# Manually bias embeddings to simulate learned relationships
# Make "cat" and "sat" closer (subject-verb)
embeddings[1] += embeddings[6] * 0.5  # cat
embeddings[6] += embeddings[1] * 0.5  # sat

# Make "chased" and "mouse" closer (verb-object)
embeddings[3] += embeddings[5] * 0.5  # chased
embeddings[5] += embeddings[3] * 0.5  # mouse

# Compute attention
W_Q = np.random.randn(d_k, d_k) * 0.1
W_K = np.random.randn(d_k, d_k) * 0.1
W_V = np.random.randn(d_k, d_k) * 0.1

Q = embeddings @ W_Q
K = embeddings @ W_K
V = embeddings @ W_V

output, weights = self_attention(Q, K, V)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(weights, cmap="Blues")
ax.set_xticks(range(n_tokens))
ax.set_yticks(range(n_tokens))
ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticklabels(tokens)
ax.set_xlabel("Key (attending to)")
ax.set_ylabel("Query (looking from)")
ax.set_title("Attention Pattern: 'The cat that chased the mouse sat down'")

for i in range(n_tokens):
    for j in range(n_tokens):
        ax.text(j, i, f"{weights[i, j]:.2f}", ha="center", va="center", fontsize=7)

plt.colorbar(im)
plt.tight_layout()
plt.show()

## Summary

| Component | Purpose |
|-----------|---------|
| Self-attention | Each token attends to all others via Q, K, V |
| Scaling (√d_k) | Prevents large dot products from saturating softmax |
| Multi-head | Multiple attention heads capture different relationships |
| Positional encoding | Adds order information to permutation-invariant attention |

**Key insight:** Attention is a soft dictionary lookup — queries find relevant keys, and retrieve their values.

## Exercises

1. Modify the self-attention function to use **causal masking** (each position can only attend to previous positions). Hint: set future positions to -infinity before softmax.
2. Experiment with different `d_k` values — how does scaling affect the attention distribution?
3. Implement **cross-attention** where Q comes from one sequence and K, V from another.